# SBND octant comparison (selected events)

Compare data vs MC in each SBND detector **octant** (E/W × N/S × Top/Bottom), following
`scripts/selected_events.py` with `--do_octant_plots`.

**Included:** octant diagnostic bar plots; per-octant topology overlays for core + final variables.

**Excluded:** per-TPC / in-TPC1 / in-TPC2 volume splits; forward/backward/crosser muon cuts; off-beam (intime) subtraction.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from os import path, makedirs
from datetime import datetime
from functools import partial
import pickle
import json
import warnings

import numpy as np
import pandas as pd

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')
from pyanalib.split_df_helpers import *
from analysis_village.numucc_1p0pi.final_selected_evt_vars import (
    CORE_SELECTED_EVT_VARIABLE_CONFIGS,
    FINAL_SELECTED_EVT_VARIABLE_CONFIGS,
)
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.files_config import *

plt.style.use("presentation.mplstyle")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# --- notebook config ---
save_fig = False          # set True to write PDFs under save_fig_dir

# Optional: override systematics bundle (None → hardcoded paths in get_syst_unc)
_SYST_RESULTS_DIR = None
_GENIE_COV_PKL = None


In [ ]:
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, f"selected_events-octants-{today_str}/Full")
save_fig_dir_octant = path.join(save_fig_dir, "by_octant")

if save_fig:
    for d in (save_fig_dir, save_fig_dir_octant):
        if not path.exists(d):
            makedirs(d)
        print("saving plots in", d)
else:
    print("save_fig=False (plots displayed inline only)")


## Load dataframes


In [ ]:
# dfs = get_ana_dfs(option="selected_events")

# mc_evt_df = dfs["mc"]
# data_evt_df = dfs["data"]
# pot_label = dfs["pot_label"]


In [ ]:
from pyanalib.split_df_helpers_new import *
pot_label = ""

filename = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/data/BNB/sel_mup-data-1e20.df"
df_data = load_dfs(filename, keys2load=['hdr', 'evt', 'bnbpot'], n_max_concat=300)
df_data.keys()

# ## -- Data
# df_dir = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_14_234844__sel_mup-data-1e20/merged_perTPC"
# keys2load_data = ['hdr', 'evt', 'bnbpot']
# df_data = dfs_from_dir(search_dir=df_dir, filename_str="sel_mup", keys2load=keys2load_data, n_max_concat=300)

data_evt_df = df_data['evt']
data_hdr_df = df_data['hdr']


In [ ]:
data_bnbpot_df = df_data['bnbpot']
# data_tot_pot = data_hdr_df['pot'].sum()
data_tot_pot = data_bnbpot_df["TOR875"].sum()

print(data_tot_pot)

In [ ]:
# TODO
# Data
data_bnbpot_df = df_data['bnbpot']
# data_tot_pot = data_hdr_df['pot'].sum()
data_tot_pot = data_bnbpot_df["TOR875"].sum()
data_evt_df["pot_weight"] = np.ones(len(data_evt_df))
print("data_tot_pot: %.3e" %(data_tot_pot))
pot_str = get_pot_str(data_tot_pot)
pot_label = f"Events / Bin (POT={pot_str})"

data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))


# nominal MC
df_dir = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_084007__sel_mup-wgts_mcstat/merged_perTPC"
keys2load_data = ['hdr', 'evt']
df_data = dfs_from_dir(search_dir=df_dir, filename_str="sel_mup", keys2load=keys2load_data, n_max_concat=300)
mc_nom_evt_df = df_data['evt']
mc_nom_hdr_df = df_data['hdr']

# DENT MCs
df_dir = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_12_112820__sel_mup-mc-EField_R00/merged_perTPC"
# df_dir = "/exp/sbnd/data/users/munjung/xsec/DENT/R00"
keys2load_data = ['hdr', 'evt']
df_data = dfs_from_dir(search_dir=df_dir, filename_str="sel_mup-mc", keys2load=keys2load_data, n_max_concat=999)
mc_R00_evt_df = df_data['evt']
mc_R00_hdr_df = df_data['hdr']

df_dir = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_12_112945__sel_mup-mc-EField_R30_Short/merged_perTPC"
# df_dir = "/exp/sbnd/data/users/munjung/xsec/DENT/R30Short"
keys2load_data = ['hdr', 'evt']
df_data = dfs_from_dir(search_dir=df_dir, filename_str="sel_mup-mc", keys2load=keys2load_data, n_max_concat=999)
mc_R30Short_evt_df = df_data['evt']
mc_R30Short_hdr_df = df_data['hdr']


mc_nom_tot_pot = mc_nom_hdr_df['pot'].sum()
mc_nom_pot_scale = data_tot_pot / mc_nom_tot_pot
print("mc_nom_pot_scale: %.3e" %(mc_nom_pot_scale))
mc_nom_evt_df["pot_weight"] = mc_nom_pot_scale * np.ones(len(mc_nom_evt_df))

mc_R00_tot_pot = mc_R00_hdr_df['pot'].sum()
mc_R00_pot_scale = data_tot_pot / mc_R00_tot_pot
print("mc_R00_pot_scale: %.3e" %(mc_R00_pot_scale))
mc_R00_evt_df["pot_weight"] = mc_R00_pot_scale * np.ones(len(mc_R00_evt_df))

mc_R30Short_tot_pot = mc_R30Short_hdr_df['pot'].sum()
mc_R30Short_pot_scale = data_tot_pot / mc_R30Short_tot_pot
print("mc_R30Short_pot_scale: %.3e" %(mc_R30Short_pot_scale))
mc_R30Short_evt_df["pot_weight"] = mc_R30Short_pot_scale * np.ones(len(mc_R30Short_evt_df))

In [ ]:
mc_evt_df = mc_nom_evt_df

In [ ]:
mc_evt_df.loc[mc_evt_df.mc.iscc.isna(), ("mc", "iscc")] = 999
data_evt_df["mc", "iscc"] = 999

# track φ in degrees (same as selected_events.py)
for df in (mc_evt_df, data_evt_df):
    df[('mu', 'pfp', 'trk', 'phi', '', '', '')] = np.degrees(
        np.arctan2(df['mu', 'pfp', 'trk', 'dir', 'x', '', ''],
                   df['mu', 'pfp', 'trk', 'dir', 'y', '', '']))
    df[('p', 'pfp', 'trk', 'phi', '', '', '')] = np.degrees(
        np.arctan2(df['p', 'pfp', 'trk', 'dir', 'x', '', ''],
                   df['p', 'pfp', 'trk', 'dir', 'y', '', '']))


In [ ]:
# POT weights and pot_label are set by get_ana_dfs(option="selected_events")
print(pot_label)
print("data events:", len(data_evt_df), "mc events:", len(mc_evt_df))


## Systematics helper


In [ ]:
def _zero_cov(var_config):
    n = len(var_config.bin_centers)
    return np.zeros((n, n))


def get_syst_unc(var_config):
    plots_base = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"

    def _cov_frac_from_npz(npz_obj, var_sn, inner_key):
        z = dict(npz_obj)
        if var_sn not in z:
            return _zero_cov(var_config)
        ret = z[var_sn].item()[inner_key]
        return ret["cov_frac"]

    if _SYST_RESULTS_DIR is not None:
        bundle = _SYST_RESULTS_DIR
        mcstat_npz = np.load(path.join(bundle, "mcstat_syst_dict.npz"), allow_pickle=True)
        g4_npz = np.load(path.join(bundle, "g4_syst_dict.npz"), allow_pickle=True)
        flux_npz = np.load(path.join(bundle, "flux_syst_dict.npz"), allow_pickle=True)
        cosmics_npz = np.load(path.join(bundle, "cosmics_syst_dict.npz"), allow_pickle=True)
    else:
        date_str = "20260220"
        mcstat_npz = np.load(path.join(plots_base, f"systematics-{date_str}", "mcstat_syst_dict.npz"), allow_pickle=True)
        g4_npz = np.load(path.join(plots_base, f"systematics-{date_str}", "g4_syst_dict.npz"), allow_pickle=True)
        flux_npz = np.load(path.join(plots_base, f"systematics-{date_str}", "flux_syst_dict.npz"), allow_pickle=True)
        date_str = "20260222"
        cosmics_npz = np.load(path.join(plots_base, f"systematics-{date_str}", "cosmics_syst_dict.npz"), allow_pickle=True)

    vn = var_config.var_save_name
    mcstat_syst = _cov_frac_from_npz(mcstat_npz, vn, "MCstat")
    g4_syst = _cov_frac_from_npz(g4_npz, vn, "G4")
    flux_syst = _cov_frac_from_npz(flux_npz, vn, "flux")
    cosmics_syst = _cov_frac_from_npz(cosmics_npz, vn, "Cosmics")

    if _GENIE_COV_PKL is not None:
        genie_path = _GENIE_COV_PKL
    else:
        genie_path = path.join(plots_base, "cov_mat_dict-20260219.pkl")
    genie_blob = pickle.load(open(genie_path, "rb"))
    try:
        genie_syst = genie_blob[var_config.var_save_name]["genie"]
    except KeyError:
        genie_syst = _zero_cov(var_config)

    pot_frac_unc = 0.02
    ntargets_frac_unc = 0.01

    frac_uncert_total = np.zeros(len(var_config.bin_centers))
    cov_total = np.zeros((len(var_config.bin_centers), len(var_config.bin_centers)))
    for syst in [mcstat_syst, genie_syst, flux_syst, g4_syst, cosmics_syst]:
        cov_total += syst
        syst_uncert = np.sqrt(np.diag(syst))
        frac_uncert_total += syst_uncert ** 2

    for syst in [pot_frac_unc, ntargets_frac_unc]:
        cov_total += np.diag(syst * np.ones(len(var_config.bin_centers)) ** 2)
        syst_uncert = syst * np.ones(len(var_config.bin_centers))
        frac_uncert_total += syst_uncert ** 2

    frac_uncert_total = np.sqrt(frac_uncert_total)
    return cov_total, frac_uncert_total


## SBND octant labeling

Split planes at ``(x0, y0, z0) = (0, 0, 250)`` cm — same convention as `selected_events.py`:
- E/W from vertex *x* (negative *x* → East)
- N/S from vertex *z* (lower *z* → South, higher *z* → North)
- Top/Bottom from vertex *y*


In [ ]:
def _weighted_counts(series, weights):
    if isinstance(series.dtype, pd.CategoricalDtype):
        cats = series.dtype.categories
        out = pd.Series(0.0, index=cats, dtype=float)
        grp = pd.DataFrame({"k": series, "w": weights}).groupby("k")["w"].sum()
        out.loc[grp.index] = grp.values
        return out
    return pd.DataFrame({"k": series, "w": weights}).groupby("k")["w"].sum().sort_index()


def _add_sbnd_octant_labels(df, x0, y0, z0):
    x = df.slc.vertex.x.astype(float)
    y = df.slc.vertex.y.astype(float)
    z = df.slc.vertex.z.astype(float)

    ew = np.where(x < x0, "E", "W")
    ns = np.where(z < z0, "S", "N")
    tb = np.where(y >= y0, "Top", "Bottom")
    octant = np.char.add(np.char.add(np.char.add(ew, "-"), np.char.add(ns, "-")), tb)

    ew = pd.Categorical(ew, categories=["W", "E"], ordered=True)
    ns = pd.Categorical(ns, categories=["S", "N"], ordered=True)
    tb = pd.Categorical(tb, categories=["Bottom", "Top"], ordered=True)
    octant = pd.Categorical(
        octant,
        categories=[
            "W-S-Bottom", "W-S-Top", "W-N-Bottom", "W-N-Top",
            "E-S-Bottom", "E-S-Top", "E-N-Bottom", "E-N-Top",
        ],
        ordered=True,
    )

    out = df.copy()
    out[("sbnd", "octant", "ew", "", "", "", "")] = ew
    out[("sbnd", "octant", "ns", "", "", "", "")] = ns
    out[("sbnd", "octant", "tb", "", "", "", "")] = tb
    out[("sbnd", "octant", "octant", "", "", "", "")] = octant
    return out


def _plot_octant_bars(df, title, save_base=None):
    w = df["pot_weight"].astype(float)
    ew = df["sbnd"].octant.ew
    ns = df["sbnd"].octant.ns
    tb = df["sbnd"].octant.tb
    oc = df["sbnd"].octant.octant

    counts_ew = _weighted_counts(ew, w)
    counts_ns = _weighted_counts(ns, w)
    counts_tb = _weighted_counts(tb, w)
    counts_oc = _weighted_counts(oc, w)

    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs = axs.flatten()

    for ax, counts, xlabel in [
        (axs[0], counts_ew, "TPC E/W (x split)"),
        (axs[1], counts_ns, "N/S (z split)"),
        (axs[2], counts_tb, "Top/Bottom (y split)"),
        (axs[3], counts_oc, "Octant (E/W, N/S, Top/Bottom)"),
    ]:
        xs = np.arange(len(counts.index))
        ax.bar(xs, counts.values, color="C0", alpha=0.85)
        ax.set_xticks(xs)
        ax.set_xticklabels([str(x) for x in counts.index], rotation=30, ha="right")
        ax.set_ylabel("Weighted Events")
        ax.set_xlabel(xlabel)
        ax.grid(True, axis="y", alpha=0.25)

    fig.suptitle(title, fontsize=18)
    fig.tight_layout(rect=[0, 0.02, 1, 0.95])
    if save_fig and save_base is not None:
        fig.savefig(save_base + fig_ext, bbox_inches="tight", dpi=dpi)
    plt.show()
    plt.close(fig)
    return {"ew": counts_ew, "ns": counts_ns, "tb": counts_tb, "octant": counts_oc}


_x0, _y0, _z0 = 0, 0, 250
print_sbnd_octant_vertex_ranges(_x0, _y0, _z0)

data_evt_df = _add_sbnd_octant_labels(data_evt_df, _x0, _y0, _z0)
mc_evt_df = _add_sbnd_octant_labels(mc_evt_df, _x0, _y0, _z0)


### Octant population (weighted event counts)


In [ ]:
_plot_octant_bars(
    data_evt_df,
    title=f"SBND octants (data) — split@({_x0:.1f},{_y0:.1f},{_z0:.1f})",
    save_base=path.join(save_fig_dir, "sbnd_octants_data") if save_fig else None,
)
_plot_octant_bars(
    mc_evt_df,
    title="SBND octants (MC)",
    save_base=path.join(save_fig_dir, "sbnd_octants_mc") if save_fig else None,
)


## Per-octant variable overlays


In [ ]:
ratio = True
approval = "internal"
textloc = [0.03, 0.55]
ax_ylim_ratio = 1.9

_SBND_OCTANT_LABELS = (
    "W-S-Bottom", "W-S-Top", "W-N-Bottom", "W-N-Top",
    "E-S-Bottom", "E-S-Top", "E-N-Bottom", "E-N-Top",
)


def _octant_rowmask(df, oct_label):
    return df["sbnd"].octant.octant.astype(str) == oct_label


chi2_records = []


### Core variables (full systematic covariance)


In [ ]:
var_configs_topology = list(CORE_SELECTED_EVT_VARIABLE_CONFIGS)

for oct_label in _SBND_OCTANT_LABELS:
    oct_slug = oct_label.replace("-", "_")
    oct_save_dir = path.join(save_fig_dir_octant, oct_slug)
    if save_fig and not path.exists(oct_save_dir):
        makedirs(oct_save_dir)

    d_o = data_evt_df.loc[_octant_rowmask(data_evt_df, oct_label)]
    m_o = mc_evt_df.loc[_octant_rowmask(mc_evt_df, oct_label)]
    if len(d_o) < 1 and len(m_o) < 1:
        print(f"skip {oct_label}: no events")
        continue

    print(f"octant {oct_label}: data={len(d_o)} mc={len(m_o)}")

    plotter_oct = partial(
        overlay_hists,
        mc_df=m_o,
        data_df=d_o,
        intime_df=None,
        dirt_df=None,
        ax_ylim_ratio=ax_ylim_ratio,
        ratio=ratio,
        textloc=textloc,
        approval=approval,
        save_fig=save_fig,
        plot=not save_fig,
    )

    for var_config in var_configs_topology:
        # cov, _ = get_syst_unc(var_config)
        for breakdown_type in ["topology"]:
            plot_labels_hist = [
                var_config.var_labels[1],
                pot_label,
                f"nominal · Octant {oct_label}",
            ]
            ret = plotter_oct(
                breakdown_type=breakdown_type,
                var_config=var_config,
                plot_labels=plot_labels_hist,
                # syst=cov,
                textchi2=True,
                save_name=path.join(
                    oct_save_dir,
                    f"{var_config.var_save_name}_{breakdown_type}",
                ),
            )
            if ret.get("chi2_val") is not None:
                chi2_records.append({
                    "var_name": var_config.var_save_name,
                    "breakdown_type": breakdown_type,
                    "cut_label": f"octant_nominal_{oct_slug}",
                    "chi2": float(ret["chi2_val"]),
                    "p_val": float(ret["p_val"]),
                    "ndof": int(ret["ndof"]),
                })


### Final-sample variables (fractional uncertainty per octant)


In [ ]:
var_configs_phi = []
_seen = set()
for _vc in FINAL_SELECTED_EVT_VARIABLE_CONFIGS:
    if _vc.var_save_name not in _seen:
        var_configs_phi.append(_vc)
        _seen.add(_vc.var_save_name)

for oct_label in _SBND_OCTANT_LABELS:
    oct_slug = oct_label.replace("-", "_")
    oct_save_dir = path.join(save_fig_dir_octant, oct_slug)
    if save_fig and not path.exists(oct_save_dir):
        makedirs(oct_save_dir)

    d_o = data_evt_df.loc[_octant_rowmask(data_evt_df, oct_label)]
    m_o = mc_evt_df.loc[_octant_rowmask(mc_evt_df, oct_label)]
    if len(d_o) < 1 and len(m_o) < 1:
        continue

    plotter_oct = partial(
        overlay_hists,
        mc_df=m_o,
        data_df=d_o,
        intime_df=None,
        dirt_df=None,
        ax_ylim_ratio=ax_ylim_ratio,
        ratio=ratio,
        textloc=textloc,
        approval=approval,
        save_fig=save_fig,
        plot=not save_fig,
    )

    for var_config in var_configs_phi:
        for breakdown_type in ["topology"]:
            frac_unc, cov = get_frac_unc(m_o, None, None, var_config)
            plot_labels_hist = [
                var_config.var_labels[1],
                pot_label,
                f"nominal · Octant {oct_label}",
            ]
            ret = plotter_oct(
                breakdown_type=breakdown_type,
                var_config=var_config,
                plot_labels=plot_labels_hist,
                syst=cov,
                textchi2=True,
                save_name=path.join(
                    oct_save_dir,
                    f"{var_config.var_save_name}_{breakdown_type}",
                ),
            )
            if ret.get("chi2_val") is not None:
                chi2_records.append({
                    "var_name": var_config.var_save_name,
                    "breakdown_type": breakdown_type,
                    "cut_label": f"octant_phi_nominal_{oct_slug}",
                    "chi2": float(ret["chi2_val"]),
                    "p_val": float(ret["p_val"]),
                    "ndof": int(ret["ndof"]),
                })


In [ ]:
chi2_df = pd.DataFrame(chi2_records)
display(chi2_df.sort_values(["cut_label", "var_name"]))

if save_fig:
    chi2_path = path.join(save_fig_dir, "chi2_records_octants.json")
    with open(chi2_path, "w") as f:
        json.dump(chi2_records, f, indent=2)
    print(f"wrote {chi2_path} ({len(chi2_records)} entries)")
